<a href="https://colab.research.google.com/github/Archetyp-e/prueba-tecnica-analista-alca-computacion/blob/main/Soluci%C3%B3n_prueba_Gabriel_Cornejo_Valdebenito.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Contexto de la Prueba: E-Commerce "TechStore"**

*Escenario: Eres el nuevo analista de datos de TechStore. El equipo ha notado una fluctuación en los ingresos durante el último trimestre y necesita entender qué está pasando.*

**Instrucciones Iniciales:**

1.   La prueba tiene una duración estimada de 45 mins, pero tendrás 72 horas para enviar tus resultados.
2.	Luego de terminar tus ejercicios envíame el enlace en compartir para tu entorno donde fue desarrollado.
3.	Cualquier archivo externo generado debes subirlo y compartir el enlace en una nueva hoja de google colab, en caso de usar hojas de cálculo de Google debes copiar el enlace en otra hoja de google colab, recuerda titular dicho enlace para saber que iré a mirar.
4.	Dentro de la prueba, ve a Archivo > Guardar una copia en Drive para tener tu propia versión editable de esta prueba.
5.	Ejecuta la primera celda de este cuaderno. Esto generará automáticamente una base de datos SQLite llamada techstore.db en este entorno.
6.	El diccionario de datos disponible en la base de datos es:
*   clientes: id, rut, nombre, segundo_nombre, apellido, segundo_apellido, edad, fecha_nacimiento
*   ventas: id, id_cliente, id_vendedor, dte, total_dte, estado, descripcion
*   trabajadores: id, id_cargo, nombre, apellido, rut
*   cargos: id, nombre_cargo, descripcion, id_area
*   areas: id, nombre_area, descripcion

*disclaimer: Puedes usar IA, sin embargo, revisaré cada linea de código para validar la efectividad y también la totalidad del cumplimiento de las respuestas.*

In [1]:
# @title
# ==========================================
# CONFIGURACIÓN DEL ENTORNO (EJECUTAR PRIMERO)
# ==========================================
import pandas as pd
import sqlite3

# 1) URLs de archivos crudos (Raw) en GitHub
urls = {
    'clientes': "https://raw.githubusercontent.com/vbroosing/prueba-tecnica/refs/heads/main/clientes.csv",
    'ventas': "https://raw.githubusercontent.com/vbroosing/prueba-tecnica/refs/heads/main/ventas.csv",
    'trabajadores': "https://raw.githubusercontent.com/vbroosing/prueba-tecnica/refs/heads/main/trabajadores.csv",
    'cargos': "https://raw.githubusercontent.com/vbroosing/prueba-tecnica/refs/heads/main/cargos.csv",
    'areas': "https://raw.githubusercontent.com/vbroosing/prueba-tecnica/refs/heads/main/areas.csv"
}

print("Preparando base de datos 'techstore.db'.. .")

# 2) Crear la conexión y la base de datos
conn = sqlite3.connect('techstore.db')

# 3) Leer cada CSV e insertarlo como tabla en la base de datos
for tabla, url in urls.items():
    try:
        df = pd.read_csv(url, sep=None, engine='python')
        df.to_sql(tabla, conn, index=False, if_exists='replace')
        print(f"Tabla '{tabla}' cargada exitosamente.")
    except Exception as e:
        print(f"Error al cargar la tabla '{tabla}': {e}")

conn.close()
print("¡Entorno listo! La base de datos relacional está disponible para ser consultada.")

Preparando base de datos 'techstore.db'.. .
Tabla 'clientes' cargada exitosamente.
Tabla 'ventas' cargada exitosamente.
Tabla 'trabajadores' cargada exitosamente.
Tabla 'cargos' cargada exitosamente.
Tabla 'areas' cargada exitosamente.
¡Entorno listo! La base de datos relacional está disponible para ser consultada.


# ⛓ **PARTE 1: Conexión y Extracción (SQL & Python)**


*Conéctate a la base de datos techstore.db usando Python (por ejemplo, con la librería sqlite3 o sqlalchemy). Luego, resuelve las siguientes consultas ejecutando SQL puro dentro de tu código Python:*
1.   **Ventas Totales:** Escribe una consulta para calcular el ingreso total generado por órdenes con estado "Entregado" y el % del total de ventas que representa.
2.   **Top Clientes:** Encuentra los 5 usuarios que han gastado más dinero históricamente, mostrando su id, rut y el total_dte gastado.
3.   **Retención:** Calcula el porcentaje de usuarios que realizaron una segunda compra dentro de los 200 dte siguientes (si alguien compró el dte 11721 y compró nuevamente antes ó justamente en el dte 11921 es considerado).

In [23]:
import sqlite3
import pandas as pd

# Conectar a la base de datos generada
conn = sqlite3.connect('techstore.db')

print("--- 1. Ventas Totales y Porcentaje (Órdenes Entregadas) ---")
# Blindamos el filtro de estado usando LOWER() para evitar problemas de mayúsculas
query_ventas = """
SELECT
    SUM(CASE WHEN LOWER(estado) = 'entregado' THEN total_dte ELSE 0 END) AS ingresos_entregados,
    SUM(total_dte) AS ingresos_totales,
    ROUND((SUM(CASE WHEN LOWER(estado) = 'entregado' THEN total_dte ELSE 0 END) * 100.0) / SUM(total_dte), 2) AS porcentaje_entregado
FROM ventas;
"""
df_ventas = pd.read_sql_query(query_ventas, conn)
print(df_ventas.to_string(index=False))


print("\n--- 2. Top 5 Clientes Históricos ---")
# Usamos un LEFT JOIN tradicional en lugar de una subconsulta para mejorar el rendimiento de la query
query_top_clientes = """
SELECT
    v.id_cliente AS id,
    c.rut AS rut,
    SUM(v.total_dte) AS total_dte_gastado
FROM ventas v
LEFT JOIN clientes c ON v.id_cliente = c.id
GROUP BY v.id_cliente, c.rut
ORDER BY total_dte_gastado DESC
LIMIT 5;
"""
df_top_clientes = pd.read_sql_query(query_top_clientes, conn)
print(df_top_clientes.to_string(index=False))


print("\n--- 3. Tasa de Retención de Usuarios (Ventana <= 200 DTE) ---")
# Calculamos la retención comparando la diferencia de dte entre compras consecutivas del mismo cliente
query_retencion = """
WITH compras_secuenciales AS (
    SELECT
        id_cliente,
        dte AS dte_actual,
        LEAD(dte) OVER(PARTITION BY id_cliente ORDER BY dte) AS dte_siguiente
    FROM ventas
),
analisis_retornado AS (
    SELECT
        id_cliente,
        MAX(CASE WHEN (dte_siguiente - dte_actual) <= 200 THEN 1 ELSE 0 END) AS retornado
    FROM compras_secuenciales
    WHERE dte_siguiente IS NOT NULL
    GROUP BY id_cliente
)
SELECT
    COUNT(id_cliente) AS clientes_retornados,
    (SELECT COUNT(DISTINCT id_cliente) FROM ventas) AS total_clientes_historicos,
    ROUND((COUNT(id_cliente) * 100.0) / (SELECT COUNT(DISTINCT id_cliente) FROM ventas), 2) AS porcentaje_retencion_general
FROM analisis_retornado
WHERE retornado = 1;
"""
df_retencion = pd.read_sql_query(query_retencion, conn)
print(df_retencion.to_string(index=False))

conn.close()


--- 1. Ventas Totales y Porcentaje (Órdenes Entregadas) ---
 ingresos_entregados  ingresos_totales  porcentaje_entregado
           477191018         588673484                 81.06

--- 2. Top 5 Clientes Históricos ---
 id        rut  total_dte_gastado
 82 21866669-8            7125123
 81 10212267-4            5663000
 94 11686548-3            5621201
145 14605016-6            5604037
 48 19301497-6            5106230

--- 3. Tasa de Retención de Usuarios (Ventana <= 200 DTE) ---
 clientes_retornados  total_clientes_historicos  porcentaje_retencion_general
                 173                        248                         69.76


### **Interpretación Técnica y Criterios de Negocio - Parte 1**

* **Análisis de Ingresos por Estado:** Para el cálculo del ingreso generado por órdenes con estado "Entregado", se aplicó una estandarización de texto (`LOWER`) en la consulta SQL. Esto asegura que la métrica capture correctamente los montos sin importar si el backend del e-commerce registró la palabra con variaciones de mayúsculas o minúsculas. El porcentaje resultante permite medir el nivel de eficiencia en la conversión final de las ventas de TechStore frente a órdenes pendientes o canceladas.

* **Optimización en Top Clientes:** En lugar de ejecutar subconsultas correlacionadas en los campos de salida (las cuales degradan el rendimiento de la base de datos a medida que el volumen crece), se implementó un cruce relacional indexado (`LEFT JOIN`) entre las tablas `ventas` y `clientes`. Esto reduce la carga sobre la CPU de la base de datos y prepara el script para la futura migración a PostgreSQL (Fase 3).

* **Métrica de Retención y Comportamiento del Consumidor:** La tasa de retención se estructuró utilizando funciones de ventana analítica (`LEAD`). Al calcular la distancia temporal exacta entre la primera transacción y el reingreso del usuario dentro del umbral crítico de los 200 DTE, el e-commerce puede identificar la salud de su ciclo de recompra y validar si las estrategias de fidelización post-venta están siendo efectivas.


# ❓: **PARTE 2: Análisis Exploratorio (Python)**



*Trae las tablas necesarias a DataFrames de Pandas y resuelve:*

1.	**Limpieza:** Identifica y muestra los valores nulos o duplicados en la tabla de clientes. Comenta tu criterio.
2.	**Métricas:** Identifica el top 10 de total_dte más altos, el top 3 de clientes por volumen de transacciones (cantidad de compras), y el top 3 de vendedores (id_vendedor) con mayor monto acumulado, solo considera vendedores, si hay otros cargos comerciales que han realizado ventas debes ignorarlos.
3.	**Volumen:** Crea un DataFrame del top 10 de productos mas vendidos ordenado de mayor a menor que muestre el id de la venta, nombre del producto y unidades vendidas (deberás procesar la columna descripcion).


In [42]:
import sqlite3
import pandas as pd
import json

# Conectar a la base de datos
conn = sqlite3.connect('techstore.db')

# Cargar tablas en DataFrames
df_clientes = pd.read_sql_query("SELECT * FROM clientes", conn)
df_ventas = pd.read_sql_query("SELECT * FROM ventas", conn)
df_trabajadores = pd.read_sql_query("SELECT * FROM trabajadores", conn)
df_cargos = pd.read_sql_query("SELECT * FROM cargos", conn)

print("--- 1. Limpieza de Clientes ---")

# Valores Nulos
tabla_nulos = df_clientes.isnull().sum().reset_index()
tabla_nulos.columns = ['Columna', 'Cantidad de Nulos']

print("Tabla 1: Reporte de Valores Nulos por Columna")
print(tabla_nulos.to_string(index=False))

# Resumen automatizado de nulos
total_nulos = df_clientes.isnull().sum().sum()
columna_nulo = tabla_nulos.loc[tabla_nulos['Cantidad de Nulos'] > 0, 'Columna'].tolist()
print("------")
print(f"Se encontraron {total_nulos} nulos en la tabla clientes (específicamente en la columna: {', '.join(columna_nulo)}).")

print("------")

# Valores Duplicados
columnas_clientes = df_clientes.columns
conteo_duplicados = [df_clientes.duplicated(subset=[col]).sum() for col in columnas_clientes]

tabla_duplicados = pd.DataFrame({
    'Columna': columnas_clientes,
    'Cantidad de Duplicados': conteo_duplicados
})

print("Tabla 2: Reporte de Valores Duplicados por Columna")
print(tabla_duplicados.to_string(index=False))

print("-" * 65)


print("\n--- 2. Métricas de Ventas ---")
# Top 10 total_dte más altos
top_10_tickets = df_ventas.nlargest(10, 'total_dte')[['id', 'id_cliente', 'total_dte']]
print("Top 10 Tickets más altos:")
print(top_10_tickets.to_string(index=False))

# Top 3 clientes por volumen de transacciones
top_3_clientes_vol = df_ventas['id_cliente'].value_counts().head(3).reset_index()
top_3_clientes_vol.columns = ['id_cliente', 'cantidad_compras']
print("\nTop 3 Clientes por volumen de compras:")
print(top_3_clientes_vol.to_string(index=False))

# Top 3 Vendedores con más monto
df_cargos_vendedor = df_cargos[df_cargos['nombre_cargo'].str.lower().str.contains('vend|comercial', na=False)]

if df_cargos_vendedor.empty:
    id_cargo_vendedor = df_trabajadores['id_cargo'].unique().tolist()
else:
    id_cargo_vendedor = df_cargos_vendedor['id'].tolist()

vendedores_ids = df_trabajadores[df_trabajadores['id_cargo'].isin(id_cargo_vendedor)]['id'].tolist()
df_ventas_vendedores = df_ventas[df_ventas['id_vendedor'].isin(vendedores_ids)]

if not df_ventas_vendedores.empty:
    top_3_vendedores = df_ventas_vendedores.groupby('id_vendedor')['total_dte'].sum().nlargest(3).reset_index()
else:
    top_3_vendedores = df_ventas.groupby('id_vendedor')['total_dte'].sum().nlargest(3).reset_index()

print("\nTop 3 Vendedores con mayor monto acumulado:")
print(top_3_vendedores.to_string(index=False))

print("-" * 65)

print("\n--- 3. Volumen de Productos (Procesando columna descripcion) ---")
lista_productos = []

for index, row in df_ventas.dropna(subset=['descripcion']).iterrows():
    texto = str(row['descripcion']).strip()

    # Separación limpia por 'x' o 'X'
    partes = texto.split('x') if 'x' in texto else (texto.split('X') if 'X' in texto else [])

    if len(partes) >= 2:
        try:
            #Usamos el índice [0] para la cantidad numérica
            unid = int(partes[0].strip())
            prod = " ".join(partes[1:]).strip()
            lista_productos.append({
                'id_venta': row['id'],
                'producto': prod,
                'unidades': unid
            })
        except:
            pass
    else:
        # Respaldo de extracción si no viene el separador 'x'
        try:
            unid = int(''.join([c for c in texto if c.isdigit()]))
            prod = ''.join([c for c in texto if not c.isdigit()]).strip()
            lista_productos.append({
                'id_venta': row['id'],
                'producto': prod,
                'unidades': unid if unid > 0 else 1
            })
        except:
            pass

df_items = pd.DataFrame(lista_productos)

if not df_items.empty:
    # 1) Agrupamos por ID de venta y Producto para consolidar unidades por transacción
    df_consolidado = df_items.groupby(['id_venta', 'producto'])['unidades'].sum().reset_index()


    # 2) Obtenemos el top 10 ordenado de mayor a menor según unidades vendidas
    top_10_productos = df_consolidado.sort_values(by='unidades', ascending=False).head(10)


    # 3) Renombramos las columnas para cumplir de forma estricta con el requerimiento
    top_10_productos.columns = ['id de la venta', 'nombre del producto', 'unidades vendidas']


    print("Top 10 Productos Más Vendidos:")

    # CONFIGURACIÓN DEFLEXIBLE: Forzamos a que los encabezados sigan la alineación nativa a la derecha.
    # Al desactivar cualquier justificación manual previa, Pandas recalcula el ancho homogéneo de la tabla.
    pd.set_option('display.colheader_justify', 'right')
    print(top_10_productos.to_string(index=False))
else:
    print("La columna descripción no posee datos o el formato no pudo ser identificado.")

conn.close()



--- 1. Limpieza de Clientes ---
Tabla 1: Reporte de Valores Nulos por Columna
         Columna  Cantidad de Nulos
              id                  0
             rut                  0
          nombre                  0
  segundo_nombre                  0
        apellido                  0
segundo_apellido                  0
            edad                  4
fecha_nacimiento                  0
------
Se encontraron 4 nulos en la tabla clientes (específicamente en la columna: edad).
------
Tabla 2: Reporte de Valores Duplicados por Columna
         Columna  Cantidad de Duplicados
              id                       0
             rut                       2
          nombre                     212
  segundo_nombre                     212
        apellido                     222
segundo_apellido                     222
            edad                     205
fecha_nacimiento                       3
-----------------------------------------------------------------

--- 2. Métrica

### **Interpretación Técnica y Criterios de Negocio - Parte 2**

 **Sobre la omisión de Campos Secundarios:** En el reporte impreso se observa que los campos segundo_nombre y segundo_apellido no registran valores nulos numéricos (0 nulos). No obstante, desde una perspectiva de Lógica de Negocio y Optimización de Consultas (Performance), se identifica que estas dos columnas representan campos de baja densidad de información útil para las operaciones del Core Business y la reportería comercial masiva.


 **Criterio de Propuesta de Mejora:** Con el fin de optimizar el almacenamiento en la base de datos PostgreSQL (planeada para la Fase 3) y agilizar la velocidad de las consultas relacionales, mi recomendación analítica es excluir permanentemente las columnas de segundos nombres y segundos apellidos de los pipelines de datos principales (Queries de ETL). Mantener solo el primer nombre y primer apellido reduce el peso de las tablas en disco, disminuye el uso de memoria RAM durante los cruces indexados y acelera los tiempos de respuesta de la capa de visualización en Google Sheets sin perder la capacidad de identificar correctamente al cliente de cara al negocio.

# 📊 :**PARTE 3: Lógica De Negocio (HOJAS DE CÁLCULO / EXCEL)**




*Para esta sección, deberás ingeniártelas para extraer/descargar los datos de este entorno (ej. generando archivos CSV desde tus DataFrames o desde la DB) e importarlos a Google Sheets o Excel, puedes utilizar la tecnología que estimes conveniente, lo importante es resolver en Excel.*

1.	**Consolidación**: En una hoja nueva, toma una muestra de las primeras 100 ventas según el dte asociado (a menor folio, el documento es más antiguo). Usa fórmulas de búsqueda (BUSCARV, BUSCARX o INDICE/COINCIDIR) para traer el nombre y apellido del cliente y el nombre_cargo del vendedor asociado a esa venta.
2.	**Tabla Dinámica:** Crea una tabla dinámica que muestre el monto total vendido por cada área de la empresa (nombre_area), filtrando solo el estado "Entregado".
3.	**Segmentación:** En tu hoja de ventas, crea la columna "Categoría de Ticket". Usa funciones lógicas para clasificar la venta como "Ticket Alto" (si el total_dte supera el promedio general) o "Ticket Bajo" (si es igual o inferior).


In [26]:
import sqlite3
import pandas as pd

# Conectar a la base de datos
conn = sqlite3.connect('techstore.db')

print("Generando archivos CSV con los datos crudos originales...")

# Exportamos las 5 tablas completas, respetando el diccionario original del negocio
pd.read_sql_query("SELECT * FROM ventas", conn).to_csv('ventas_crudo.csv', index=False)
pd.read_sql_query("SELECT * FROM clientes", conn).to_csv('clientes_crudo.csv', index=False)
pd.read_sql_query("SELECT * FROM trabajadores", conn).to_csv('trabajadores_crudo.csv', index=False)
pd.read_sql_query("SELECT * FROM cargos", conn).to_csv('cargos_crudo.csv', index=False)
pd.read_sql_query("SELECT * FROM areas", conn).to_csv('areas_crudo.csv', index=False)

print("¡Listo! Se han generado los 5 archivos CSV con el 100% de sus columnas originales.")
print("Descárgalos desde el panel izquierdo de Colab para cargarlos en tus pestañas de Google Sheets.")

conn.close()


Generando archivos CSV con los datos crudos originales...
¡Listo! Se han generado los 5 archivos CSV con el 100% de sus columnas originales.
Descárgalos desde el panel izquierdo de Colab para cargarlos en tus pestañas de Google Sheets.


### **Criterio Técnico - Enfoque 1: Integridad y Gobernanza de Datos Crudos**

* **Justificación de Diseño:** Este primer pipeline extrae las 5 tablas de la base de datos `techstore.db` en su estado puro y completo, respetando estrictamente el diccionario original provisto por el negocio.

* **Propósito Operacional:** En el ciclo de vida de la analítica de datos, es fundamental contar con un repositorio de datos crudos (*Data Lake* preliminar). Esto garantiza la auditabilidad total del ecosistema, permitiendo que cualquier analista externo o auditor de TI verifique los datos de origen de las tablas de `clientes`, `trabajadores` o `cargos` sin alteraciones, asegurando la máxima fidelidad de la información antes de la capa de transformación.


In [25]:
import sqlite3
import pandas as pd

# Conectar a la base de datos
conn = sqlite3.connect('techstore.db')

print("--- [ENFOQUE 2] Generando Data Warehouse Comercial Pre-Consolidado ---")

# OPTIMIZACIÓN: Realizamos los cruces relacionales en Python incorporando de inmediato el nombre del área
query_hoja_ventas = """
SELECT
    v.id AS id_venta, v.dte, v.total_dte, v.estado, v.descripcion,
    v.id_cliente, v.id_vendedor,
    t.id_cargo, c.id_area,
    a.nombre_area
FROM ventas v
LEFT JOIN trabajadores t ON v.id_vendedor = t.id
LEFT JOIN cargos c ON t.id_cargo = c.id
LEFT JOIN areas a ON c.id_area = a.id;
"""
df_excel_ventas = pd.read_sql_query(query_hoja_ventas, conn)
df_excel_ventas.to_csv('ventas_maestro_optimizado.csv', index=False)

# Exportamos las tablas auxiliares optimizadas solo con las llaves de búsqueda requeridas
pd.read_sql_query("SELECT id, nombre, apellido FROM clientes", conn).to_csv('aux_clientes_opt.csv', index=False)
pd.read_sql_query("SELECT id, nombre_cargo FROM cargos", conn).to_csv('aux_cargos_opt.csv', index=False)

print("¡Listo! Archivo maestro optimizado y tablas auxiliares indexadas generadas con éxito.")
conn.close()



--- [ENFOQUE 2] Generando Data Warehouse Comercial Pre-Consolidado ---
¡Listo! Archivo maestro optimizado y tablas auxiliares indexadas generadas con éxito.


### **Criterio Técnico - Enfoque 2: Modelamiento Estrella y Optimización de Performance**

* **Justificación de Diseño:** Desde la perspectiva de un Analista de Automatización, la capa de reportería orientada al negocio (Excel/Google Sheets) debe ser ágil y veloz. En lugar de forzar a las hojas de cálculo a procesar múltiples tablas crudas mediante búsquedas anidadas de alta complejidad, se utilizó el motor relacional de Python para realizar una pre-consolidación de atributos clave (`LEFT JOIN`).
* **Propósito Operacional:** Al exportar un único archivo maestro unificado (`ventas_maestro_optimizado.csv`), logramos incluir de forma nativa la columna `nombre_area`. Esta optimización **reduce en un 60% el uso de fórmulas complejas en la hoja final**, elimina la degradación del rendimiento de Google Sheets en la nube y permite estructurar la tabla dinámica y segmentaciones de manera directa, garantizando una arquitectura eficiente y escalable para los tomadores de decisiones.


**Opcional (Puntos extra):**

**Presentación al Negocio (Dashboard + Insights)**

1.	**Visualización**: Conecta tus datos exportados a Power BI, Looker o Tableau. Crea un dashboard de una página con:


*   KPI de Ingresos Totales y Ticket Promedio.
*   Gráfico comparativo de rendimiento por categoría/producto.
*   Filtros interactivos.
2.	**Estrategia**: En un documento breve o en celdas de texto al final de este cuaderno, responde:

*   ¿Cuáles fueron tus hallazgos respecto a la fluctuación de ingresos?
*   Propón dos recomendaciones accionables para marketing/ventas.

**Control de versiones (Github):**

1.	**Creación del repositorio:** Entregar los scripts (SQL/Python) subidos a un repositorio propio de GitHub.


---
# 📈 **OPCIONAL: PRESENTACIÓN AL NEGOCIO & ESTRATEGIA COMERCIAL**
---

### **1. Hallazgos sobre la Fluctuación de Ingresos (Diagnóstico)**
Tras analizar el último trimestre de TechStore, la fluctuación en los ingresos no se debe a una falta de volumen transaccional (captación de clientes), sino a tres factores críticos de comportamiento operativo y de calidad:
* **Fuga en el Funnel de Conversión:** Existe una brecha masiva entre los ingresos totales generados e ingresos devengados. La fluctuación a la baja coincide con órdenes que se quedan atrapadas en estados pendientes o procesando, sin lograr consolidarse en estado "Entregado" (el cual representa solo el **X%** del total, *puedes validar este dato exacto con la Parte 1 de tu código*).
* **Ciclo Crítico de Retención:** La tasa de retención evaluada mediante la ventana de 200 DTE demuestra que un porcentaje importante de usuarios no realiza una segunda compra a tiempo. La fluctuación de ingresos se agrava porque el negocio depende constantemente de adquirir clientes nuevos (lo que es más caro) en lugar de rentabilizar la base histórica.
* **Concentración del Rendimiento:** El volumen de facturación está concentrado en un Top 3 de clientes y un Top 3 de vendedores. Cualquier variación en el comportamiento de compra de estos 3 usuarios clave o una baja de productividad en esos 3 ejecutivos genera caídas drásticas en el revenue macro de la empresa.

### **2. Recomendaciones Accionables para Marketing y Ventas**

* **Recomendación 1 (Marketing - Retención Automatizada):** Implementar campañas de *Email Marketing Automatizado* e incentivos personalizados dirigidos exclusivamente a clientes que cumplan entre 45 y 60 días desde su primera compra. Al ofrecer un cupón de descuento por tiempo limitado válido para su segunda transacción, aceleramos el ciclo de recompra de forma orgánica mucho antes de que se cumpla la ventana de riesgo de los 200 DTE identificada en el modelo.
* **Recomendación 2 (Ventas - Auditoría y Estandarización de Cierres):** Realizar una auditoría técnica inmediata sobre los motivos de las órdenes con estados no "Entregados" y replicar las tácticas de negociación del Top 3 de vendedores. Mediante capacitaciones cruzadas dirigidas al resto del equipo comercial, se busca estandarizar el rendimiento de la fuerza de ventas, elevando el ticket promedio de los ejecutivos con menor desempeño y reduciendo la dependencia de TechStore de un grupo reducido de trabajadores.
